# Orchid Classification - ResNet18

Pipeline completa per la classificazione di orchidee del genere *Ophrys*.

**Autore originale:** Davide Di Pierro (UNINA)

---

**Flusso:** Configurazione → Preprocessing → Training → Testing → Inference

## 1. Configurazione

Eseguire questa cella prima di qualsiasi altra operazione.

In [ ]:
# ============================================
# IMPORT
# ============================================
import os
import random
import shutil
import torch
from torch.utils.data import DataLoader

from app.timing import Timer
from app.preprocessing_tools.reproducibility import set_seed

# ============================================
# PATHS
# ============================================
RAW_DATA_PATH = 'dataset/raw/'
AUGMENTED_DATA_PATH = 'dataset/augmented/'
PROCESSED_DATA_TRAIN_PATH = 'dataset/processed/processed_data_train.pt'
PROCESSED_DATA_VALIDATION_PATH = 'dataset/processed/processed_data_validation.pt'
PROCESSED_DATA_TEST_PATH = 'dataset/processed/processed_data_test.pt'
MODEL_PATH = 'models/model.pt'

# ============================================
# CLASSI
# ============================================
CLASS_NAMES = [
    'O. exaltata',
    'O. garganica',
    'O. incubacea',
    'O. majellensis',
    'O. sphegodes',
    'O. sphegodes_Palena'
]
CLASS_SIZE = len(CLASS_NAMES)

# ============================================
# PARAMETRI PREPROCESSING (CPU)
# ============================================
WIDTH = 256
HEIGHT = 512
TRAIN_SIZE = 0.7
VALIDATION_SIZE = 0.1  # TEST_SIZE = 1 - TRAIN_SIZE - VALIDATION_SIZE
DYNAMIC_AUGMENTATION = False
BATCH_SIZE_PREPROCESS = 32
NUM_WORKERS_PREPROCESS = 16
SHOULD_NORMALIZE = True

# ============================================
# PARAMETRI TRAINING (GPU)
# ============================================
FORCE_CPU = False
BATCH_SIZE_TRAIN = 8
NUM_WORKERS_TRAIN = 4
EPOCHS_TRAIN = 26
LEARNING_RATE = 0.001
PRETRAINED = True

# ============================================
# PARAMETRI TESTING
# ============================================
BATCH_SIZE_TEST = 32
NUM_WORKERS_TEST = 8
SLIDING_WINDOW_SIZE = 30
SLIDING_WINDOW_STRIDE = 15
DESTINATION_PATH = 'results/'

# ============================================
# PARAMETRI INFERENCE
# ============================================
PATH_TO_PREPROCESS = 'inference/raw/'
INFERENCE_PATH = 'inference/test/'
DESTINATION_PATH_INFERENCE = 'inference/results/'

# ============================================
# RIPRODUCIBILITÀ
# ============================================
SEED = 57
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
os.environ["PYTHONHASHSEED"] = "0"
torch.use_deterministic_algorithms(True)
set_seed(SEED)

# ============================================
# NORMALIZZAZIONE (calcolata sul dataset)
# ============================================
MEAN = '0.538217306137085 0.5433580875396729 0.368904173374176'
STD = '0.20864470303058624 0.23085597157478333 0.2029736042022705'

# ============================================
# DEVICE SELECTION (MPS > CUDA > CPU)
# ============================================
def get_device(force_cpu=False):
    if force_cpu:
        return torch.device('cpu')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    elif torch.cuda.is_available():
        return torch.device('cuda')
    else:
        return torch.device('cpu')

print(f"Configurazione caricata. Classi: {CLASS_SIZE}")
print(f"Device: {get_device(FORCE_CPU)}")

---

## 2. Preprocessing

### 2.1 Augmentation, Splitting e Resize

Divide le immagini in train/validation/test e applica data augmentation.

In [ ]:
from app.preprocess_data import DynamicAugmentationWrapper
from app.preprocessing_tools import rotation

if __name__ == '__main__':
    with Timer('Augmentation'):
        work_dir = 'work'
        augmentations = [
            rotation.identity,
            rotation.horizontal_and_vertical_flip,
            rotation.horizontal_flip,
            rotation.vertical_flip
        ]
        
        DynamicAugmentationWrapper(
            RAW_DATA_PATH,
            work_dir,
            AUGMENTED_DATA_PATH,
            CLASS_NAMES,
            TRAIN_SIZE,
            VALIDATION_SIZE,
            WIDTH,
            HEIGHT,
            DYNAMIC_AUGMENTATION,
            augmentations
        )
        
        if os.path.exists(work_dir):
            shutil.rmtree(work_dir)
            print(f'Removed directory {work_dir}')
        
        print('Augmentation completata')

### 2.2 Calcolo MEAN e STD

Ricalcola i valori di normalizzazione sul dataset augmentato. Aggiornare i valori nella cella di configurazione se necessario.

In [ ]:
from app.preprocess_data import fresh_normalization

if __name__ == '__main__':
    with Timer('Normalization'):
        mean, std = fresh_normalization(
            AUGMENTED_DATA_PATH + 'train/',
            WIDTH,
            HEIGHT,
            BATCH_SIZE_PREPROCESS,
            NUM_WORKERS_PREPROCESS,
            SEED
        )
        MEAN = ' '.join([str(x) for x in mean])
        STD = ' '.join([str(x) for x in std])
        print(f'MEAN: {MEAN}')
        print(f'STD: {STD}')
        print('\nAggiorna i valori nella cella di configurazione se diversi!')

### 2.3 Conversione in Tensori

Converte le immagini in tensori e salva su disco.

In [ ]:
from app.preprocess_data import preprocessData

PATHS = [
    AUGMENTED_DATA_PATH + 'train/',
    AUGMENTED_DATA_PATH + 'validation/',
    AUGMENTED_DATA_PATH + 'test/'
]

if __name__ == '__main__':
    with Timer('Preprocessing'):
        print('Preprocessing training dataset...')
        preprocessData(
            PATHS[0], PROCESSED_DATA_TRAIN_PATH,
            WIDTH, HEIGHT, BATCH_SIZE_PREPROCESS, NUM_WORKERS_PREPROCESS,
            SHOULD_NORMALIZE, SEED, MEAN, STD
        )
        
        print('Preprocessing validation dataset...')
        preprocessData(
            PATHS[1], PROCESSED_DATA_VALIDATION_PATH,
            WIDTH, HEIGHT, BATCH_SIZE_PREPROCESS, NUM_WORKERS_PREPROCESS,
            SHOULD_NORMALIZE, SEED, MEAN, STD
        )
        
        print('Preprocessing test dataset...')
        preprocessData(
            PATHS[2], PROCESSED_DATA_TEST_PATH,
            WIDTH, HEIGHT, BATCH_SIZE_PREPROCESS, NUM_WORKERS_PREPROCESS,
            SHOULD_NORMALIZE, SEED, MEAN, STD
        )
        
        print('Preprocessing completato')

---

## 3. Training

In [ ]:
from app.train_model import loadAndTrain, saveModel

if __name__ == '__main__':
    with Timer('Training'):
        device = get_device(FORCE_CPU)
        
        if device == torch.device('cpu'):
            print('WARNING: Running on CPU. Training will be slow.')
        
        print(f'Device: {device}')
        
        model = loadAndTrain(
            device, CLASS_SIZE,
            PROCESSED_DATA_TRAIN_PATH,
            PROCESSED_DATA_VALIDATION_PATH,
            BATCH_SIZE_TRAIN,
            EPOCHS_TRAIN,
            NUM_WORKERS_TRAIN,
            LEARNING_RATE,
            PRETRAINED,
            MODEL_PATH,
            SEED
        )
        
        saveModel(model, MODEL_PATH)
        print('Training completato')

---

## 4. Testing

In [ ]:
from app.test_model import deviceLoader, modelLoader, testModel, generateOutputImages
from app.preprocessing_tools.dataset_tool import getDatasetFromFile

if __name__ == '__main__':
    with Timer('Testing'):
        device = deviceLoader()
        model = modelLoader(MODEL_PATH, CLASS_SIZE, device)
        
        testDataset = getDatasetFromFile(PROCESSED_DATA_TEST_PATH)
        testDataLoader = DataLoader(
            testDataset,
            BATCH_SIZE_TEST,
            shuffle=True,
            num_workers=NUM_WORKERS_TEST,
            prefetch_factor=2,
            persistent_workers=True
        )
        
        testModel(model, testDataLoader, device, CLASS_NAMES, CLASS_SIZE)
        
        generateOutputImages(
            testDataset, model, device,
            CLASS_NAMES, DESTINATION_PATH,
            SLIDING_WINDOW_SIZE, SLIDING_WINDOW_STRIDE
        )
        
        print('Testing completato')

---

## 5. Inference

Inferenza su nuove immagini non etichettate.

### 5.1 Preprocessing immagini

In [ ]:
from app.preprocessing_tools.dataset_tool import augmentDataPath
from app.preprocessing_tools import rotation

if __name__ == '__main__':
    with Timer('Inference Preprocessing'):
        augmentations = [rotation.identity]
        
        if os.path.exists(PATH_TO_PREPROCESS):
            if any(os.path.isdir(os.path.join(PATH_TO_PREPROCESS, f)) 
                   for f in os.listdir(PATH_TO_PREPROCESS)):
                for folder in os.listdir(PATH_TO_PREPROCESS):
                    folder_path = os.path.join(PATH_TO_PREPROCESS, folder)
                    if os.path.isdir(folder_path):
                        augmentDataPath(
                            folder_path,
                            INFERENCE_PATH + folder + '/',
                            100000, WIDTH, HEIGHT,
                            augmentations
                        )
            else:
                augmentDataPath(
                    PATH_TO_PREPROCESS,
                    INFERENCE_PATH,
                    100000, WIDTH, HEIGHT,
                    augmentations
                )
            print('Preprocessing inference completato')
        else:
            print(f'Cartella {PATH_TO_PREPROCESS} non trovata')

### 5.2 Inferenza

In [ ]:
from app.inference import loadDevice, loadModel, generateOutputImages
from app.preprocess_data import getTransforms
from app.preprocessing_tools.dataset_tool import SingleFolderDataset

if __name__ == '__main__':
    with Timer('Inference'):
        device = loadDevice(FORCE_CPU)
        model = loadModel(MODEL_PATH, CLASS_SIZE, device)
        model.eval()
        
        test_dataset = SingleFolderDataset(
            INFERENCE_PATH,
            transform=getTransforms(WIDTH, HEIGHT, SHOULD_NORMALIZE, MEAN, STD)
        )
        
        generateOutputImages(
            test_dataset, model, device,
            CLASS_NAMES, DESTINATION_PATH_INFERENCE,
            SLIDING_WINDOW_SIZE, SLIDING_WINDOW_STRIDE
        )
        
        print('Inference completata')